# Cashew Leaf Disease Classification

A deep-learning image classification notebook for detecting cashew leaf disease categories using transfer learning, class-weighted optimization, and multi-model evaluation.

## What this notebook demonstrates

- Preparing the CCMT Cashew subset for five-class disease classification.
- Exploring class imbalance and visualizing disease-category distribution.
- Applying stratified train/validation/test splitting.
- Computing class weights for weighted cross-entropy loss.
- Applying data augmentation and ImageNet normalization.
- Fine-tuning EfficientNet-B0, ResNet-18, VGG-16, and DenseNet-121 using PyTorch and timm.
- Evaluating models with accuracy, macro F1-score, per-class metrics, and confusion matrices.

## Local dataset layout

Place the dataset locally using this structure, or update `DATASET_ROOT` in the notebook:

```text
data/raw/Cashew/
├── anthracnose/
├── gumosis/
├── leaf miner/
├── red rust/
└── healthy/
```



# EDA & Data Preparation (splits + class weights)


In [ ]:
# Optional: install missing dependency if needed
# pip install timm


In [ ]:
from pathlib import Path
p = Path("data/raw/Cashew")
print("Classes:", [d.name for d in p.iterdir() if d.is_dir()])


In [ ]:
# Optional for Google Colab users:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
!rm -rf "outputs/splits"


In [ ]:
# Optional Colab extraction example.
# Update the zip path and output path before running.
# !unzip -o -q "data/Dataset for Crop Pest and Disease Detection.zip" \
#   "Dataset for Crop Pest and Disease Detection/Raw Data/CCMT Dataset/Cashew/*" \
#   -d data/raw


# Dataset
Download the CCMT dataset from Mendeley Data and place the Cashew subset under `data/raw/Cashew/`.


In [ ]:
import os

for root, dirs, files in os.walk("/content/CCMT_raw", topdown=True):
    print(root)


In [ ]:
from pathlib import Path

# Path of dataset
DATASET_ROOT = Path("data/raw/Cashew")

# Output file
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLASSES = ['anthracnose', 'gumosis', 'leaf miner', 'red rust', 'healthy']

print("Dataset ready at:", DATASET_ROOT)
print("Classes:", [d.name for d in DATASET_ROOT.iterdir() if d.is_dir()])


# Scan data + Count categories + Draw the distribution + Save table


In [ ]:
import os, random, csv, json
from collections import Counter, defaultdict
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
from sklearn.model_selection import train_test_split
import shutil
from pathlib import Path
import matplotlib.pyplot as plt

import numpy as np, torch, random
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Collection of routes and addresses
rows = []
for c in CLASSES:
    cls_dir = DATASET_ROOT / c
    if not cls_dir.exists():
        raise RuntimeError(f"The folder does not exist: {cls_dir}")
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            rows.append({"path": str(cls_dir / fname), "label": c})

if not rows:
    raise RuntimeError("No images")

df = pd.DataFrame(rows)
counts = Counter(df["label"].tolist())
total = sum(counts.values())
print("Total images:", total)
counts


In [ ]:
# Save the distribution table as a CSV file
counts_table = pd.DataFrame(
    [{"Class": c, "Count": counts[c], "Percent": 100 * counts[c] / total} for c in CLASSES]
)
counts_table.to_csv(OUTPUT_DIR / "class_counts_full.csv", index=False)
counts_table


In [ ]:
# Vertical distribution diagram
plt.figure(figsize=(8,4))
plt.bar(CLASSES, [counts[c] for c in CLASSES])
plt.title("Class Distribution (Raw Counts)")
plt.ylabel("#Images")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution_full.png", dpi=200)
plt.show()


### Custom Augmentation Pipeline
We define a custom augmentation function that applies random horizontal flips, rotations, and color jitter. This helps improve model robustness and generalization on unseen data.


In [ ]:
def simple_augment(im):
    # 50% chance horizontal flip
    if random.random() < 0.5:
        im = ImageOps.mirror(im)
    # random rotation between -15° and +15°
    angle = random.uniform(-15, 15)
    im = im.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=None)
    # slight color, brightness, and contrast variations
    im = ImageEnhance.Brightness(im).enhance(random.uniform(0.9, 1.1))
    im = ImageEnhance.Contrast(im).enhance(random.uniform(0.9, 1.1))
    im = ImageEnhance.Color(im).enhance(random.uniform(0.9, 1.1))
    return im

# sample 5 random images, show original + 3 augmented versions
sample_df = df.sample(min(5, len(df)), random_state=SEED)
nrows, ncols = len(sample_df), 4

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(3*ncols, 3*nrows))
if nrows == 1:  # handle single-row case
    axes = [axes]

for r, (_, row) in enumerate(sample_df.iterrows()):
    im = Image.open(row["path"]).convert("RGB")
    axes[r][0].imshow(im)
    axes[r][0].set_title(f"Original\n{row['label']}")
    axes[r][0].axis("off")
    for c in range(1, ncols):
        aug = simple_augment(im.copy())
        axes[r][c].imshow(aug)
        axes[r][c].set_title("Augmented")
        axes[r][c].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "augmentation_preview.png", dpi=200)
plt.show()
print("Saved preview: augmentation_preview.png")


### Stratified Data Split


In [ ]:
# Split into train and temp (val + test) using stratified sampling
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)

# Split temp into val and test (15% / 15% of the original data)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)

# Function to copy images into organized folders
def copy_split(split_df, split_name):
    for _, row in split_df.iterrows():
        src = Path(row["path"])
        cls = row["label"]
        dst_dir = OUTPUT_DIR / "splits" / split_name / cls
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst_dir / src.name)

# Copy each split
copy_split(train_df, "train")
copy_split(val_df, "val")
copy_split(test_df, "test")

# Show number of images in each split
len(train_df), len(val_df), len(test_df)


In [ ]:
# Count the categories
counts_train = Counter(train_df["label"].tolist())
counts_val   = Counter(val_df["label"].tolist())
counts_test  = Counter(test_df["label"].tolist())

print("Train:", counts_train)
print("Val  :", counts_val)
print("Test :", counts_test)


### Handling Imbalance: Class Weighting
Since the dataset is imbalanced (e.g., Gummosis is a minority class), we calculate Class Weights inversely proportional to class frequencies. These weights will be passed to the Loss function to penalize errors on minority classes more heavily.


In [ ]:
# class_weights
counts_train = Counter(train_df["label"].tolist())

N = sum(counts_train.values())
K = len(CLASSES)
weights = {c: float(N / (K * max(1, counts_train[c]))) for c in CLASSES}

with open(OUTPUT_DIR / "class_weights.json", "w", encoding="utf-8") as f:
    json.dump(weights, f, indent=2)


# Training and Evaluation (EfficientNet-B0)


In [ ]:
# Training
import json, time, torch, torch.nn as nn, timm, numpy as np, pandas as pd
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt, seaborn as sns
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("outputs")
SPLITS = ROOT/"splits"
assert SPLITS.exists(), f"Splits folder not found: {SPLITS}"

# Data & Transforms
IMG_SIZE = 224
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])
train_ds = datasets.ImageFolder(SPLITS/"train", transform=train_tfms)
val_ds   = datasets.ImageFolder(SPLITS/"val",   transform=val_tfms)
test_ds  = datasets.ImageFolder(SPLITS/"test",  transform=val_tfms)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

# Class weights
with open(ROOT/"class_weights.json") as f:
    wdict = json.load(f)
cls_weights = torch.tensor([wdict[c] for c in train_ds.classes], dtype=torch.float32, device=DEVICE)

def build_model(arch, num_classes):
    return timm.create_model(arch, pretrained=True, num_classes=num_classes).to(DEVICE)

def run_one_experiment(arch, lr, epochs=30, es_patience=5):
    tag = f"{arch}_lr{lr:g}"
    model = build_model(arch, len(train_ds.classes))
    criterion = nn.CrossEntropyLoss(weight=cls_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Checkpoints
    best_ckpt   = ROOT / f"best_{tag}.pt"
    resume_ckpt = ROOT / f"resume_{tag}.pt"

    best_f1 = 0.0
    start_epoch = 1
    no_improve = 0

    if resume_ckpt.exists():
        r = torch.load(resume_ckpt, map_location=DEVICE)
        model.load_state_dict(r["model"])
        optimizer.load_state_dict(r["optim"])
        scheduler.load_state_dict(r["sched"])
        start_epoch = r["epoch"] + 1
        best_f1 = r.get("best_f1", 0.0)
        print(f"[{tag}] Resuming from epoch {start_epoch}")

    hist = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "val_f1":[]}

    def _epoch(loader, train=False):
        model.train() if train else model.eval()
        tot=correct=loss_sum=0.0; preds=[]; trues=[]
        for x,y in tqdm(loader, leave=False, desc="Train" if train else "Eval"):
            x,y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            if train:
                loss.backward(); optimizer.step()
            tot += y.size(0); loss_sum += loss.item()*y.size(0)
            p = out.argmax(1)
            correct += (p==y).sum().item()
            preds += p.detach().cpu().tolist()
            trues += y.detach().cpu().tolist()
        f1 = f1_score(trues, preds, average="macro", zero_division=0)
        return loss_sum/tot, correct/tot, f1

    start_time = time.time()
    for e in range(start_epoch, epochs+1):
        tl, ta, _   = _epoch(train_loader, train=True)
        vl, va, vf1 = _epoch(val_loader,   train=False)
        scheduler.step()

        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_acc"].append(ta); hist["val_acc"].append(va); hist["val_f1"].append(vf1)
        print(f"[{tag}] Epoch {e:02d}/{epochs} TL={tl:.3f} TA={ta:.3f} | VL={vl:.3f} VA={va:.3f} F1={vf1:.3f}")

        # Saving the best weight based on the best Val F1 + Improvement Counter
        if vf1 > best_f1:
            best_f1 = vf1
            no_improve = 0
            torch.save({k: v.cpu() for k, v in model.state_dict().items()}, best_ckpt)
        else:
            no_improve += 1

        # # Save the checkpoint for each epoch
        torch.save({
            "model": model.state_dict(),
            "optim": optimizer.state_dict(),
            "sched": scheduler.state_dict(),
            "epoch": e,
            "best_f1": best_f1
        }, resume_ckpt)

        # Early Stopping
        if no_improve >= es_patience:
            print(f"[{tag}] Early stopping triggered at epoch {e} (patience={es_patience})")
            break

    duration = time.time() - start_time

    # Save the training date and fees
    pd.DataFrame(hist).to_csv(ROOT/f"history_{tag}.csv", index=False)
    plt.figure(); plt.plot(hist["train_loss"],label="Train"); plt.plot(hist["val_loss"],label="Val"); plt.legend(); plt.title(f"Loss - {tag}"); plt.tight_layout()
    plt.savefig(ROOT/f"loss_{tag}.png", dpi=200); plt.close()
    plt.figure(); plt.plot(hist["train_acc"],label="Train"); plt.plot(hist["val_acc"],label="Val"); plt.legend(); plt.title(f"Accuracy - {tag}"); plt.tight_layout()
    plt.savefig(ROOT/f"acc_{tag}.png", dpi=200); plt.close()

    return {"tag":tag, "arch":arch, "lr":lr, "epochs":epochs,
            "best_val_f1":round(best_f1,4),
            "time_min":round(duration/60,1),
            "best_ckpt":str(best_ckpt)}

# The experiences
CONFIGS = [
    {"arch":"efficientnet_b0", "lr":3e-4, "epochs":30},
    {"arch":"efficientnet_b0", "lr":1e-4, "epochs":30},
    {"arch":"resnet18",        "lr":3e-4, "epochs":30},
    {"arch":"resnet18",        "lr":1e-4, "epochs":30},
    {"arch":"vgg16",           "lr":3e-4, "epochs":30},
    {"arch":"densenet121",     "lr":3e-4, "epochs":30},
]

results = []
for cfg in CONFIGS:
    print("\n" + "="*80)
    print(f"Running: {cfg}")
    out = run_one_experiment(**cfg)
    results.append(out)

df_res = pd.DataFrame(results).sort_values(by=["best_val_f1"], ascending=False)
display(df_res)
df_res.to_csv(ROOT/"experiments_summary_phase2.csv", index=False)
print("Saved:", ROOT/"experiments_summary_phase2.csv")


---


# Model Evaluation Setup
We evaluate the model using Macro F1-Score alongside Accuracy.

Macro F1 is the preferred metric here because it treats all classes equally, regardless of their size in the dataset.


In [ ]:
# Evaluation & Reporting
import json, torch, numpy as np, pandas as pd
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("outputs")
SPLITS = ROOT/"splits"
assert SPLITS.exists(), f"Splits folder not found: {SPLITS}"

# The same transforms used in val/test
IMG_SIZE = 224
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

test_ds  = datasets.ImageFolder(SPLITS/"test",  transform=val_tfms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
CLASSES = test_ds.classes

def build_model(arch, num_classes):
    import timm, torch.nn as nn
    model = timm.create_model(arch, pretrained=False, num_classes=num_classes)
    return model.to(DEVICE)

def evaluate_checkpoint(tag, arch):
    best_ckpt = ROOT / f"best_{tag}.pt"
    assert best_ckpt.exists(), f"Best checkpoint not found: {best_ckpt}"

    model = build_model(arch, len(CLASSES))
    state = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(state)

    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for x,y in test_loader:
            x = x.to(DEVICE)
            logits = model(x)
            preds = logits.argmax(1).detach().cpu().numpy().tolist()
            all_preds += preds
            all_trues += y.numpy().tolist()

    # Metrics
    acc  = accuracy_score(all_trues, all_preds)
    f1_m = f1_score(all_trues, all_preds, average="macro", zero_division=0)
    rep  = classification_report(all_trues, all_preds, target_names=CLASSES,
                                 output_dict=True, zero_division=0)
    df_rep = pd.DataFrame(rep).transpose()
    per_class_csv = ROOT / f"per_class_metrics_{tag}.csv"
    df_rep.to_csv(per_class_csv, index=True)

    # Confusion Matrices (count + row-normalized)
    cm = confusion_matrix(all_trues, all_preds, labels=list(range(len(CLASSES))))
    cm_png = ROOT / f"cm_{tag}.png"
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion - {tag}")
    plt.tight_layout(); plt.savefig(cm_png, dpi=220); plt.close()

    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    cmn_png = ROOT / f"cm_normalized_{tag}.png"
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion (Row-Normalized) - {tag}")
    plt.tight_layout(); plt.savefig(cmn_png, dpi=220); plt.close()

    return {
        "tag": tag,
        "test_acc": round(float(acc), 4),
        "test_f1_macro": round(float(f1_m), 4),
        "per_class_csv": str(per_class_csv),
        "cm_png":  str(cm_png),
        "cmn_png": str(cmn_png),
    }

CONFIGS = [
    {"arch":"efficientnet_b0", "lr":3e-4, "epochs":30},
    {"arch":"efficientnet_b0", "lr":1e-4, "epochs":30},
    {"arch":"resnet18",        "lr":3e-4, "epochs":30},
    {"arch":"resnet18",        "lr":1e-4, "epochs":30},
    {"arch":"vgg16",           "lr":3e-4, "epochs":30},
    {"arch":"densenet121",     "lr":3e-4, "epochs":30},
]

# Evaluate each experience
eval_rows = []
for cfg in CONFIGS:
    tag = f"{cfg['arch']}_lr{cfg['lr']:g}"
    print(f"Evaluating: {tag}")
    row = evaluate_checkpoint(tag, cfg["arch"])
    eval_rows.append(row)

df_phase3 = pd.DataFrame(eval_rows).sort_values(by=["test_f1_macro","test_acc"], ascending=False)
display(df_phase3)
df_phase3.to_csv(ROOT/"experiments_summary_phase3.csv", index=False)
print("Saved:", ROOT/"experiments_summary_phase3.csv")


---


# Results & Analysis Dashboard


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

ROOT = Path("outputs")
if not ROOT.exists():
    print("Outputs missing. Please run training cells first.")

def read_csv_safe(name):
    p = ROOT / name
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

# Load experiment summaries (training summary and test summary)
train_summary = read_csv_safe("experiments_summary_phase2.csv")
test_summary  = read_csv_safe("experiments_summary_phase3.csv")

# Show training summary
print("## Training Summary")
if train_summary.empty:
    display(HTML("<b style='color:#b00'>No training summary CSV found.</b>"))
else:
    display(train_summary.sort_values("best_val_f1", ascending=False).reset_index(drop=True))

# Show test summary
print("\n## Test Evaluation")
if test_summary.empty:
    display(HTML("<b style='color:#b00'>No test summary CSV found.</b>"))
else:
    display(test_summary.sort_values(["test_f1_macro","test_acc"], ascending=False).reset_index(drop=True))

# Combined view on 'tag'
print("\n## Combined View (joined on tag)")
if not train_summary.empty and not test_summary.empty:
    combined = pd.merge(train_summary, test_summary, on="tag", how="left")
    sort_cols = [c for c in ["test_f1_macro", "test_acc", "best_val_f1"] if c in combined.columns]
    combined = combined.sort_values(sort_cols, ascending=False).reset_index(drop=True)
    display(combined)
else:
    display(HTML("<i>Combined view requires both summaries.</i>"))

# Quick bar chart: Val F1 vs Test F1 per tag
print("\n## F1 Comparison (Validation vs Test)")
have_val = (not train_summary.empty) and ("best_val_f1" in train_summary.columns)
have_test = (not test_summary.empty) and ("test_f1_macro" in test_summary.columns)
if have_val or have_test:
    tags = sorted(set(train_summary["tag"].tolist() if have_val else []) |
                  set(test_summary["tag"].tolist()  if have_test else []))
    val_map  = dict(zip(train_summary["tag"], train_summary["best_val_f1"])) if have_val else {}
    test_map = dict(zip(test_summary["tag"],  test_summary["test_f1_macro"])) if have_test else {}

    val_vals  = [val_map.get(t, None) for t in tags]
    test_vals = [test_map.get(t, None) for t in tags]

    x = range(len(tags))
    width = 0.35
    plt.figure(figsize=(10, 4))
    if have_val:
        plt.bar([i - width/2 for i in x], val_vals, width, label="Val F1 (best)")
    if have_test:
        plt.bar([i + width/2 for i in x], test_vals, width, label="Test F1 (macro)")
    plt.xticks(list(x), tags, rotation=20, ha="right")
    plt.ylim(0, 1.0)
    plt.ylabel("F1")
    plt.title("Val vs Test F1 per Experiment")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    display(HTML("<i>No F1 values available to plot.</i>"))

# Pick best experiment (prefer Test F1, then Val F1) and display details
def pick_best_tag():
    if not test_summary.empty:
        return test_summary.sort_values(["test_f1_macro","test_acc"], ascending=False).iloc[0]["tag"]
    if not train_summary.empty:
        return train_summary.sort_values("best_val_f1", ascending=False).iloc[0]["tag"]
    return None

best_tag = pick_best_tag()
if best_tag:
    print(f"\n## Best Experiment: {best_tag}")

    # Per-class metrics
    per_class_csv = ROOT / f"per_class_metrics_{best_tag}.csv"
    if per_class_csv.exists():
        df_pc = pd.read_csv(per_class_csv)
        display(HTML("<b>Per-class metrics (precision / recall / f1-score / support)</b>"))
        display(df_pc)

    # Confusion matrices (count + normalized)
    cm_path  = ROOT / f"cm_{best_tag}.png"
    cmn_path = ROOT / f"cm_normalized_{best_tag}.png"
    if cm_path.exists():
        display(HTML("<b>Confusion Matrix</b>"))
        display(Image(filename=str(cm_path)))
    if cmn_path.exists():
        display(HTML("<b>Confusion Matrix (Row-Normalized)</b>"))
        display(Image(filename=str(cmn_path)))
else:
    display(HTML("<b style='color:#b00'>No experiment found to highlight as best.</b>"))


# Project Conclusion

## Project Overview & Methodology
This project presents a deep learning system for classifying Cashew leaf diseases into five categories.  
To address dataset imbalance—particularly the minority Gummosis class—we applied stratified splitting (70% Train, 15% Validation, 15% Test) and computed class-balanced weights for the loss function.

The experimentation pipeline was based on Transfer Learning using four architectures:

- **EfficientNet-B0**
- **ResNet-18**
- **VGG-16**
- **DenseNet-121**

All models were trained using the AdamW optimizer and a Cosine Annealing learning rate schedule to ensure stable convergence and strong generalization.

## Key Results
All four architectures achieved strong performance on both validation and test sets.  
The best-performing model was EfficientNet-B0 (lr = 3e-4), achieving:

- Test Accuracy: 98.58%  
- Test Macro F1-Score: 98.80%

The model demonstrated excellent classification capability across all classes, including the minority class Gummosis.  

## Future Work
Future directions for expanding this project include:

- Applying Grad-CAM for interpretability and visualizing model attention.  
- Exploring advanced augmentation strategies such as MixUp or CutMix to further enhance robustness.
